# 11 — Proximal Policy Optimization (PPO)

## Learning Objectives
1. Understand the PPO-Clip objective and why it prevents destabilising policy updates
2. Implement GAE (Generalised Advantage Estimation) for lower-variance gradient estimates
3. Train an actor-critic agent on a hand-coded CartPole environment using multiple epochs per rollout
4. Explore how clip_epsilon, K epochs, and GAE lambda affect sample efficiency and stability


In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import time
from typing import Tuple, List, Dict

np.random.seed(42)

print("NumPy version:", np.__version__)
print("Imports OK — no gym/torch needed")


## Level 1: PPO Clip Objective — Core Mechanics

The PPO-Clip objective is:

L_CLIP(theta) = E[min(r_t(theta) * A_t, clip(r_t(theta), 1-epsilon, 1+epsilon) * A_t)]

where r_t = pi_theta(a|s) / pi_old(a|s) is the probability ratio.

Here we isolate the clipping logic in pure numpy to build intuition.


In [ ]:
# --- Level 1: Policy ratio clipping in numpy ---

def ppo_clip_objective(ratio: np.ndarray, advantage: float, eps: float = 0.2) -> np.ndarray:
    """
    Compute PPO-Clip objective for an array of policy ratios.
    Args:
        ratio     : array of r_t = pi_new(a|s) / pi_old(a|s) values
        advantage : scalar A_t (can be positive or negative)
        eps       : clipping range, typical 0.1-0.3
    Returns:
        array of clipped objective values
    """
    clipped_ratio = np.clip(ratio, 1.0 - eps, 1.0 + eps)
    # Take element-wise minimum to pessimistically bound the objective
    return np.minimum(ratio * advantage, clipped_ratio * advantage)


# Visualise clipping for positive and negative advantages
ratios = np.linspace(0.4, 2.0, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, adv, label in zip(axes, [1.0, -1.0], ["Positive Advantage (+1)", "Negative Advantage (-1)"]):
    pg_obj = ratios * adv                                    # plain PG (no clip)
    ppo_obj = ppo_clip_objective(ratios, adv, eps=0.2)       # PPO-Clip
    ax.plot(ratios, pg_obj,  "--", label="PG (no clip)", color="steelblue")
    ax.plot(ratios, ppo_obj, "-",  label="PPO-Clip (eps=0.2)", color="darkred", linewidth=2)
    ax.axvline(0.8, color="gray", linestyle=":", alpha=0.7)
    ax.axvline(1.2, color="gray", linestyle=":", alpha=0.7, label="Clip boundaries")
    ax.axvline(1.0, color="black", linestyle="-", alpha=0.3)
    ax.set_xlabel("Policy Ratio r_t")
    ax.set_ylabel("Objective")
    ax.set_title(label)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("PPO Clip: prevents large policy updates in both directions", fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/ppo_clip.png", dpi=80, bbox_inches="tight")
plt.show()
print("Clip at r=1.5, adv=+1 → PG:", 1.5, " PPO:", ppo_clip_objective(np.array([1.5]), 1.0)[0])
print("Clip at r=0.5, adv=-1 → PG:", -0.5, "PPO:", ppo_clip_objective(np.array([0.5]), -1.0)[0])


## Level 2: Full PPO-Clip with GAE on CartPole

We implement:
- A linear softmax actor (policy)
- A linear critic (value function)
- GAE(gamma=0.99, lambda=0.95) for advantage estimation
- K=4 gradient epochs per rollout batch
- Entropy bonus to encourage exploration


In [ ]:
# === CartPole environment (no gym) ===

def cartpole_step(state, action, dt=0.02):
    """One step of CartPole physics. Returns (next_state, reward, done)."""
    x, x_dot, theta, theta_dot = state
    force = 10.0 if action == 1 else -10.0
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    temp = (force + 0.05 * theta_dot**2 * sin_t) / 1.1
    theta_acc = (9.8 * sin_t - cos_t * temp) / (0.5 * (4/3 - 0.1 * cos_t**2 / 1.1))
    x_acc = temp - 0.05 * theta_acc * cos_t / 1.1
    x = x + dt * x_dot
    x_dot = x_dot + dt * x_acc
    theta = theta + dt * theta_dot
    theta_dot = theta_dot + dt * theta_acc
    done = bool(abs(x) > 2.4 or abs(theta) > 0.2)
    return np.array([x, x_dot, theta, theta_dot]), float(not done), done


def cartpole_reset(rng):
    return rng.uniform(-0.05, 0.05, size=4)


# === Linear softmax policy ===

class LinearPolicy:
    """Softmax policy pi(a|s) = softmax(Ws + b)."""
    def __init__(self, obs_dim, n_actions):
        self.W = np.zeros((obs_dim, n_actions))
        self.b = np.zeros(n_actions)

    def probs(self, s):
        lg = s @ self.W + self.b
        lg = lg - lg.max()
        p = np.exp(lg); return p / p.sum()

    def log_prob(self, s, a):
        return np.log(self.probs(s)[a] + 1e-8)

    def entropy(self, s):
        p = self.probs(s)
        return -np.sum(p * np.log(p + 1e-8))

    def sample(self, s, rng):
        return int(rng.choice(len(self.probs(s)), p=self.probs(s)))


class LinearValue:
    """Linear baseline V(s) = w^T s + b."""
    def __init__(self, obs_dim, lr=1e-2):
        self.w = np.zeros(obs_dim); self.b = 0.0; self.lr = lr

    def predict(self, s):
        return float(self.w @ s + self.b)

    def update(self, S, targets):
        preds = S @ self.w + self.b
        err = preds - targets
        self.w -= self.lr * S.T @ err / len(err)
        self.b -= self.lr * err.mean()


def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """Generalised Advantage Estimation."""
    T = len(rewards)
    adv = np.zeros(T)
    gae = 0.0
    for t in reversed(range(T)):
        nv = values[t+1] if t+1 < len(values) else 0.0
        delta = rewards[t] + gamma * nv * (1 - float(dones[t])) - values[t]
        gae = delta + gamma * lam * (1 - float(dones[t])) * gae
        adv[t] = gae
    returns = adv + np.array(values[:T])
    return adv, returns


def ppo_grad(states, actions, old_lps, advantages, policy, clip_eps, ent_coef):
    """PPO-Clip gradient (returns grad_W, grad_b, mean_loss)."""
    T = len(states)
    gW = np.zeros_like(policy.W)
    gb = np.zeros_like(policy.b)
    total_loss = 0.0
    for i in range(T):
        s, a = states[i], int(actions[i])
        probs = policy.probs(s)
        new_lp = np.log(probs[a] + 1e-8)
        ratio = np.exp(new_lp - old_lps[i])
        clipped = np.clip(ratio, 1 - clip_eps, 1 + clip_eps)
        adv = advantages[i]
        obj = min(ratio * adv, clipped * adv)
        ent = policy.entropy(s)
        total_loss -= (obj + ent_coef * ent)
        # Policy gradient signal for clipped vs unclipped
        use_clip = (ratio * adv > clipped * adv)
        d_new_lp = -(adv if not use_clip else 0.0) - ent_coef * 0.0
        d_logits = -probs.copy()
        d_logits[a] += 1.0
        d_logits *= d_new_lp
        gW += np.outer(s, d_logits)
        gb += d_logits
    return gW / T, gb / T, total_loss / T


def train_ppo_full(n_iters=100, n_steps=512, k_epochs=4,
                   clip_eps=0.2, gamma=0.99, lam=0.95,
                   policy_lr=3e-3, value_lr=1e-2, ent_coef=0.01, seed=42):
    """Full PPO training on CartPole. Returns per-iteration episode returns."""
    rng = np.random.default_rng(seed)
    policy = LinearPolicy(4, 2)
    vf = LinearValue(4, lr=value_lr)
    history = []

    for itr in range(n_iters):
        # Collect rollout
        st_list, ac_list, r_list, d_list, lp_list, v_list = [], [], [], [], [], []
        s = cartpole_reset(rng)
        for _ in range(n_steps):
            v = vf.predict(s)
            a = policy.sample(s, rng)
            lp = policy.log_prob(s, a)
            ns, r, done = cartpole_step(s, a)
            st_list.append(s.copy()); ac_list.append(a); r_list.append(r)
            d_list.append(done); lp_list.append(lp); v_list.append(v)
            s = cartpole_reset(rng) if done else ns

        S = np.array(st_list); A = np.array(ac_list)
        old_lps = np.array(lp_list)
        adv, rets = compute_gae(r_list, v_list, d_list, gamma, lam)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        for _ in range(k_epochs):
            gW, gb, _ = ppo_grad(S, A, old_lps, adv, policy, clip_eps, ent_coef)
            policy.W -= policy_lr * gW
            policy.b -= policy_lr * gb
            vf.update(S, rets)

        history.append(sum(r_list))

    return history


print("Training PPO on CartPole (no gym)...")
t0 = time.time()
ppo_returns = train_ppo_full(n_iters=100, n_steps=512, k_epochs=4)
print(f"Done in {time.time()-t0:.1f}s | Last-20 avg: {np.mean(ppo_returns[-20:]):.1f}")


## Real-World Example 1: RLHF Simulation

PPO is the backbone of RLHF (Reinforcement Learning from Human Feedback), used to align LLMs.
Here we simulate the setup: a tiny 10-vocab language model generates sequences, a linear
reward model scores them, and PPO updates the policy with a KL penalty to prevent
reward hacking (the model shouldn't drift too far from its reference/SFT policy).


In [ ]:
# === RLHF Simulation: toy LM + reward model + KL penalty ===

class ToyLanguageModel:
    """
    10-vocab token-level LM represented as a linear policy per timestep.
    State = current token (one-hot), action = next token.
    """
    def __init__(self, vocab=10, rng=None):
        self.vocab = vocab
        self.rng = rng or np.random.default_rng(7)
        # Logits matrix: state-token -> next-token distribution
        self.logits = np.zeros((vocab, vocab))

    def probs(self, state_token):
        """Softmax distribution over next tokens."""
        lg = self.logits[state_token]
        lg = lg - lg.max()
        p = np.exp(lg); return p / p.sum()

    def sample(self, state_token):
        return int(self.rng.choice(self.vocab, p=self.probs(state_token)))

    def log_prob(self, state_token, next_token):
        return np.log(self.probs(state_token)[next_token] + 1e-8)


def linear_reward_model(token_seq, weights):
    """
    Simple reward: dot product of token frequencies with learned weights.
    Simulates a reward model trained on human preferences.
    """
    freq = np.bincount(token_seq, minlength=len(weights)) / len(token_seq)
    return float(freq @ weights)


def rlhf_ppo_step(lm, ref_lm, reward_weights, kl_coef=0.1, lr=5e-3, seq_len=8):
    """
    One PPO step on the toy LM with KL penalty.
    Returns (reward, kl_div).
    """
    rng = lm.rng
    # Sample a sequence from current policy
    seq = [rng.integers(lm.vocab)]
    for _ in range(seq_len - 1):
        seq.append(lm.sample(seq[-1]))
    seq = np.array(seq)

    # Compute extrinsic reward and KL penalty
    r_ext = linear_reward_model(seq[1:], reward_weights)

    kl = 0.0
    log_prob_policy = 0.0
    log_prob_ref = 0.0
    for t in range(len(seq) - 1):
        s, a = seq[t], seq[t+1]
        lp_pi = lm.log_prob(s, a)
        lp_ref = ref_lm.log_prob(s, a)
        log_prob_policy += lp_pi
        log_prob_ref += lp_ref
        kl += lp_pi - lp_ref

    # Shaped reward = extrinsic - kl_coef * KL divergence
    r_total = r_ext - kl_coef * kl

    # Simple REINFORCE update (PPO ratio = 1 for first step)
    for t in range(len(seq) - 1):
        s, a = seq[t], seq[t+1]
        probs = lm.probs(s)
        d_logits = -probs.copy()
        d_logits[a] += 1.0
        lm.logits[s] += lr * r_total * d_logits  # gradient ascent

    return r_ext, kl


# Setup
rng_rlhf = np.random.default_rng(42)
ref_lm = ToyLanguageModel(vocab=10, rng=rng_rlhf)
lm = ToyLanguageModel(vocab=10, rng=np.random.default_rng(99))
lm.logits = ref_lm.logits.copy()  # initialise from reference (simulates SFT)

# Reward weights: tokens 7-9 are "preferred" by reward model
reward_weights = np.array([-0.5, -0.3, -0.1, 0.0, 0.1, 0.1, 0.2, 0.5, 0.7, 0.9])

# Train with and without KL penalty
rewards_with_kl, rewards_no_kl = [], []
kl_history = []

lm_no_kl = ToyLanguageModel(vocab=10, rng=np.random.default_rng(77))
lm_no_kl.logits = ref_lm.logits.copy()

for step in range(200):
    r_kl, kl = rlhf_ppo_step(lm, ref_lm, reward_weights, kl_coef=0.1)
    r_nokl, _ = rlhf_ppo_step(lm_no_kl, ref_lm, reward_weights, kl_coef=0.0)
    rewards_with_kl.append(r_kl)
    rewards_no_kl.append(r_nokl)
    kl_history.append(kl)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
window = 20
def smooth(x): return np.convolve(x, np.ones(window)/window, mode='valid')
axes[0].plot(smooth(rewards_with_kl), label="With KL penalty", color="darkgreen")
axes[0].plot(smooth(rewards_no_kl), "--", label="No KL penalty", color="firebrick")
axes[0].set_title("RLHF: Reward over Training"); axes[0].set_xlabel("Step"); axes[0].legend()
axes[1].plot(smooth(kl_history), color="purple")
axes[1].set_title("KL Divergence from Reference Policy")
axes[1].set_xlabel("Step"); axes[1].set_ylabel("KL div")
plt.tight_layout(); plt.savefig("/tmp/rlhf.png", dpi=80); plt.show()
print(f"Final rewards — with KL: {np.mean(rewards_with_kl[-20:]):.3f}  no KL: {np.mean(rewards_no_kl[-20:]):.3f}")
print("KL penalty prevents reward hacking (policy drifting to extreme outputs)")


## Real-World Example 2: PPO on Continuous Actions (Pendulum)

Many real RL tasks have continuous action spaces (robot joints, steering angles).
PPO extends naturally: the policy becomes a Gaussian pi(a|s) = N(mu(s), sigma^2).
We implement a simple 1D pendulum stabilisation task in pure numpy.

Pendulum physics: theta'' = -(3g/2l)*sin(theta) + (u / (m*l^2))


In [ ]:
# === Continuous PPO on Pendulum (numpy, no gym) ===

def pendulum_step(state, u, dt=0.05):
    """
    1D pendulum dynamics.
    state = [theta, theta_dot]
    u = torque (continuous action, clipped to [-2, 2])
    """
    g, l, m = 9.8, 1.0, 1.0
    theta, theta_dot = state
    u = np.clip(u, -2.0, 2.0)
    theta_ddot = -(3 * g / (2 * l)) * np.sin(theta) + u / (m * l**2)
    theta_dot = theta_dot + dt * theta_ddot
    theta = theta + dt * theta_dot
    # Reward: penalise angle from upright and large torques
    reward = -(theta**2 + 0.1 * theta_dot**2 + 0.001 * u**2)
    done = abs(theta) > np.pi
    return np.array([theta, theta_dot]), reward, done


class GaussianPolicy:
    """
    Gaussian policy: mu = W_mu @ s + b_mu, log_std is a learnable parameter.
    """
    def __init__(self, obs_dim):
        self.W_mu = np.zeros((obs_dim, 1))
        self.b_mu = np.zeros(1)
        self.log_std = np.array([-0.5])  # start with some exploration

    def mean(self, s):
        return float(self.W_mu.T @ s + self.b_mu)

    def std(self):
        return float(np.exp(self.log_std))

    def sample(self, s, rng):
        return self.mean(s) + self.std() * rng.standard_normal()

    def log_prob(self, s, a):
        mu = self.mean(s); sigma = self.std()
        return -0.5 * ((a - mu) / sigma)**2 - np.log(sigma) - 0.5 * np.log(2 * np.pi)


def train_ppo_continuous(n_iters=80, n_steps=256, k_epochs=3,
                         clip_eps=0.2, gamma=0.99, lr=1e-3, seed=42):
    """PPO on pendulum with Gaussian policy."""
    rng = np.random.default_rng(seed)
    policy = GaussianPolicy(obs_dim=2)
    vf = LinearValue(obs_dim=2, lr=5e-3)
    ep_returns = []

    for itr in range(n_iters):
        st_l, ac_l, r_l, d_l, lp_l, v_l = [], [], [], [], [], []
        s = np.array([rng.uniform(-0.3, 0.3), 0.0])  # random start angle

        for _ in range(n_steps):
            v = vf.predict(s)
            a = policy.sample(s, rng)
            lp = policy.log_prob(s, a)
            ns, r, done = pendulum_step(s, a)
            st_l.append(s.copy()); ac_l.append(a); r_l.append(r)
            d_l.append(done); lp_l.append(lp); v_l.append(v)
            s = np.array([rng.uniform(-0.3, 0.3), 0.0]) if done else ns

        S = np.array(st_l); A = np.array(ac_l); old_lps = np.array(lp_l)
        adv, rets = compute_gae(r_l, v_l, d_l, gamma=gamma)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        for _ in range(k_epochs):
            for i in range(n_steps):
                new_lp = policy.log_prob(S[i], A[i])
                ratio = np.exp(new_lp - old_lps[i])
                clipped = np.clip(ratio, 1 - clip_eps, 1 + clip_eps)
                obj = min(ratio * adv[i], clipped * adv[i])
                # Gradient for Gaussian mean
                mu, sigma = policy.mean(S[i]), policy.std()
                d_mu = (A[i] - mu) / sigma**2 * obj * lr
                policy.W_mu += d_mu * S[i].reshape(-1, 1)
                policy.b_mu += d_mu
            vf.update(S, rets)

        ep_returns.append(sum(r_l) / max(1, sum(d_l)))

    return ep_returns


print("Training PPO (Gaussian policy) on Pendulum...")
t0 = time.time()
pend_returns = train_ppo_continuous(n_iters=80, n_steps=256)
print(f"Done in {time.time()-t0:.1f}s | Last-20 avg reward: {np.mean(pend_returns[-20:]):.2f}")


## Real-World Example 3: PPO Hyperparameter Sweep

PPO has three key hyperparameters that practitioners tune:
- **clip_epsilon** (eps): how much the policy ratio is allowed to change (0.1-0.3)
- **K epochs**: how many gradient steps per rollout batch (2-10)
- **GAE lambda**: bias-variance tradeoff in advantage estimation (0.9-0.99)

We sweep clip_eps x K_epochs and visualise the interaction.


In [ ]:
# === PPO Hyperparameter Sweep ===

def quick_ppo(clip_eps=0.2, k_epochs=4, lam=0.95, n_iters=60, n_steps=256, seed=42):
    """Quick PPO run; returns final-10 average return."""
    rng = np.random.default_rng(seed)
    policy = LinearPolicy(4, 2)
    vf = LinearValue(4, lr=1e-2)
    ep_returns = []

    for _ in range(n_iters):
        st_l, ac_l, r_l, d_l, lp_l, v_l = [], [], [], [], [], []
        s = cartpole_reset(rng)
        for _ in range(n_steps):
            v = vf.predict(s)
            a = policy.sample(s, rng)
            lp = policy.log_prob(s, a)
            ns, r, done = cartpole_step(s, a)
            st_l.append(s.copy()); ac_l.append(a); r_l.append(r)
            d_l.append(done); lp_l.append(lp); v_l.append(v)
            s = cartpole_reset(rng) if done else ns

        S = np.array(st_l); A = np.array(ac_l); old_lps = np.array(lp_l)
        adv, rets = compute_gae(r_l, v_l, d_l, gamma=0.99, lam=lam)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        for _ in range(k_epochs):
            gW, gb, _ = ppo_grad(S, A, old_lps, adv, policy, clip_eps, ent_coef=0.01)
            policy.W -= 3e-3 * gW
            policy.b -= 3e-3 * gb
            vf.update(S, rets)

        ep_returns.append(sum(r_l))

    return np.mean(ep_returns[-10:])


print("Running PPO hyperparameter sweep (clip_eps x K_epochs)...")
print("(This may take ~30 seconds)
")

eps_vals = [0.1, 0.2, 0.3]
k_vals = [2, 4, 8]
results = np.zeros((len(eps_vals), len(k_vals)))

t0 = time.time()
for i, eps in enumerate(eps_vals):
    for j, k in enumerate(k_vals):
        results[i, j] = quick_ppo(clip_eps=eps, k_epochs=k)
        print(f"  eps={eps}, K={k} -> {results[i,j]:.1f}")

print(f"
Sweep completed in {time.time()-t0:.1f}s")

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(results, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(k_vals))); ax.set_xticklabels([f"K={k}" for k in k_vals])
ax.set_yticks(range(len(eps_vals))); ax.set_yticklabels([f"eps={e}" for e in eps_vals])
ax.set_title("PPO Hyperparameter Sweep\nFinal-10 Cumulative Returns")
for i in range(len(eps_vals)):
    for j in range(len(k_vals)):
        ax.text(j, i, f"{results[i,j]:.0f}", ha="center", va="center", fontsize=11, fontweight="bold")
plt.colorbar(im, ax=ax, label="Return")
plt.tight_layout(); plt.savefig("/tmp/ppo_sweep.png", dpi=80); plt.show()
best = np.unravel_index(results.argmax(), results.shape)
print(f"Best config: eps={eps_vals[best[0]]}, K={k_vals[best[1]]} -> {results[best]:.1f}")


## Comparison: Policy Gradient vs PPO vs PPO+GAE

We compare three algorithms on CartPole to see how PPO's design choices
improve sample efficiency and stability:
- **REINFORCE**: plain policy gradient, high variance
- **PPO (no GAE)**: clip + MC returns, moderate variance
- **PPO+GAE**: full PPO with lambda-return advantages


In [ ]:
# === Comparison: PG vs PPO vs PPO+GAE ===

def train_reinforce(n_iters=100, n_steps=256, lr=3e-3, gamma=0.99, seed=42):
    """REINFORCE (vanilla policy gradient) baseline."""
    rng = np.random.default_rng(seed)
    policy = LinearPolicy(4, 2)
    history = []

    for _ in range(n_iters):
        states, actions, rewards, dones = [], [], [], []
        s = cartpole_reset(rng)
        for _ in range(n_steps):
            a = policy.sample(s, rng)
            ns, r, done = cartpole_step(s, a)
            states.append(s.copy()); actions.append(a); rewards.append(r); dones.append(done)
            s = cartpole_reset(rng) if done else ns

        # Monte Carlo returns
        T = len(rewards); G = np.zeros(T)
        g = 0.0
        for t in reversed(range(T)):
            g = rewards[t] + gamma * g * (1 - float(dones[t])); G[t] = g
        G = (G - G.mean()) / (G.std() + 1e-8)

        # Single gradient step
        for i in range(T):
            s, a = states[i], int(actions[i])
            probs = policy.probs(s)
            d_logits = -probs.copy(); d_logits[a] += 1.0
            policy.W -= lr * G[i] * np.outer(s, d_logits)
            policy.b -= lr * G[i] * d_logits

        history.append(sum(rewards))
    return history


def train_ppo_no_gae(n_iters=100, n_steps=256, k_epochs=4, clip_eps=0.2,
                     gamma=0.99, lr=3e-3, seed=42):
    """PPO-Clip with MC returns (no GAE)."""
    rng = np.random.default_rng(seed)
    policy = LinearPolicy(4, 2)
    vf = LinearValue(4, lr=1e-2)
    history = []

    for _ in range(n_iters):
        st_l, ac_l, r_l, d_l, lp_l = [], [], [], [], []
        s = cartpole_reset(rng)
        for _ in range(n_steps):
            a = policy.sample(s, rng); lp = policy.log_prob(s, a)
            ns, r, done = cartpole_step(s, a)
            st_l.append(s.copy()); ac_l.append(a); r_l.append(r); d_l.append(done); lp_l.append(lp)
            s = cartpole_reset(rng) if done else ns

        # MC returns as advantages (no GAE)
        T = len(r_l); rets = np.zeros(T)
        g = 0.0
        for t in reversed(range(T)):
            g = r_l[t] + gamma * g * (1 - float(d_l[t])); rets[t] = g
        adv = (rets - rets.mean()) / (rets.std() + 1e-8)

        S = np.array(st_l); A = np.array(ac_l); old_lps = np.array(lp_l)
        for _ in range(k_epochs):
            gW, gb, _ = ppo_grad(S, A, old_lps, adv, policy, clip_eps, 0.01)
            policy.W -= lr * gW; policy.b -= lr * gb
            vf.update(S, rets)

        history.append(sum(r_l))
    return history


print("Training three algorithms on CartPole...")
n_iters = 100; seeds = [42, 43, 44]  # 3 seeds for variance estimate

reinforce_runs = [train_reinforce(n_iters, seed=s) for s in seeds]
ppo_no_gae_runs = [train_ppo_no_gae(n_iters, seed=s) for s in seeds]
ppo_gae_runs = [train_ppo_full(n_iters, seed=s) for s in seeds]

def smooth_mean_std(runs):
    arr = np.array(runs)
    w = 10
    def sm(r): return np.convolve(r, np.ones(w)/w, 'valid')
    smoothed = np.array([sm(r) for r in arr])
    return smoothed.mean(0), smoothed.std(0)

x = np.arange(n_iters - 9)
fig, ax = plt.subplots(figsize=(10, 5))
for label, runs, col in [
    ("REINFORCE", reinforce_runs, "steelblue"),
    ("PPO (no GAE)", ppo_no_gae_runs, "orange"),
    ("PPO + GAE", ppo_gae_runs, "darkgreen"),
]:
    mu, std = smooth_mean_std(runs)
    ax.plot(x, mu, label=label, color=col)
    ax.fill_between(x, mu - std, mu + std, alpha=0.2, color=col)

ax.set_xlabel("Iteration"); ax.set_ylabel("Cumulative Return")
ax.set_title("CartPole: REINFORCE vs PPO vs PPO+GAE (mean +/- std over 3 seeds)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("/tmp/ppo_comparison.png", dpi=80); plt.show()
print(f"Final-10 means -> REINFORCE: {np.mean(reinforce_runs[0][-10:]):.1f} | "
      f"PPO (no GAE): {np.mean(ppo_no_gae_runs[0][-10:]):.1f} | "
      f"PPO+GAE: {np.mean(ppo_gae_runs[0][-10:]):.1f}")


## Key Takeaways

**Core idea:** PPO adds a clipping constraint to vanilla policy gradient, preventing the
policy from changing too drastically in a single update. This improves stability without
requiring the complex trust-region calculation of TRPO.

### Variants and When to Use

| Method | Advantage Estimate | When to Use | Trade-off |
|--------|--------------------|-------------|-----------|
| REINFORCE | MC returns | Simple discrete tasks | High variance |
| PPO (MC) | MC returns + clip | When simplicity matters | Moderate variance |
| PPO + GAE | GAE(lambda) + clip | Most production use | Lambda tuning needed |
| PPO (RLHF) | Reward model + KL | LLM alignment | Reward hacking risk |

### Common Failure Modes

- **clip_eps too large (>0.3):** Policy changes too much per update, unstable training
  -> Reduce clip_eps to 0.1-0.2
- **Too many K epochs:** Policy diverges from old policy, ratio r_t becomes unreliable
  -> Use K=4-8; check ratio magnitudes during training
- **GAE lambda too high (1.0):** Becomes MC estimation; high variance, slow convergence
  -> Use lambda=0.9-0.97 for best bias-variance tradeoff
- **Missing entropy bonus:** Policy collapses to deterministic too early
  -> Add entropy coefficient 0.01-0.05

### Related Concepts

- [10-policy-gradient](./10-policy-gradient.ipynb) — PPO builds directly on REINFORCE
- [12-soft-actor-critic](./12-soft-actor-critic.ipynb) — alternative on-policy vs off-policy
- [15-reward-shaping](./15-reward-shaping.ipynb) — shapes the reward signal PPO optimises


## Exercises

1. **Modify clip_eps:** Change epsilon from 0.2 to 0.05 and observe how conservatively
   the policy updates each iteration. Plot the ratio r_t distribution over training.

2. **Implement value clipping:** Add a clipped value loss analogous to the policy clip:
   V_loss = max((V - V_old)^2, (clip(V, V_old-eps, V_old+eps) - V_t)^2)
   Does this stabilise training?

3. **RLHF ablation:** In the RLHF cell, increase kl_coef from 0.1 to 0.5 and observe
   how the policy stays closer to the reference at the cost of lower reward.

4. **Continuous action clip:** The Gaussian PPO example clips actions. Try removing
   the action clip (allow |u| > 2) — does training destabilise?

5. **GAE sensitivity:** In `compute_gae`, sweep lambda over [0.5, 0.8, 0.9, 0.95, 1.0].
   Plot the variance of advantages. At what lambda does variance become too high?
